In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('../../..')

In [3]:
is_hitter = True

In [4]:
from Model.Constants import DATA_PREP_BINARY_FILE
from Model.Combined.DataPrep.Data_Prep import Combined_Data_Prep

data_prep = Combined_Data_Prep.Load_From_File("../../" + DATA_PREP_BINARY_FILE)

In [5]:
io_list = data_prep.Generate_IO_Hitters(is_training=True) if is_hitter else data_prep.Generate_IO_Pitchers(is_training=True)

In [6]:
from Model.Combined.Tuning.ProWar import objective, ProModelTuningRecipe
from functools import partial

recipe = ProModelTuningRecipe.RECURRENT | ProModelTuningRecipe.BATCH_PARAMS
objective_func = partial(
    objective,
    io_list=io_list,
    data_prep=data_prep,
    is_hitter=is_hitter,
    recipe=recipe
)

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

study_name = "Pro_Hitter_WAR_Tuning" if is_hitter else "Pro_Pitcher_WAR_Tuning"
study_name += f"_LSTM_{recipe.name}"
#optuna.delete_study(study_name=study_name, storage="sqlite:///Pro_Tune.db")
study = optuna.create_study(
    direction="minimize",
    load_if_exists=True,
    study_name=study_name,
    storage="sqlite:///Pro_Tune.db"
)

In [8]:
study.optimize(objective_func, n_trials=100, show_progress_bar=True)

  0%|          | 0/100 [00:00<?, ?it/s]

[W 2026-07-22 19:44:34,903] Trial 70 failed with parameters: {'num_layers': 4, 'hidden_size': 32, 'dropout': 0.2505540040212866, 'wd_shared': 0.06585034743240242, 'lr_shared': 0.001060663895339256, 'batch_size': 1022, 'num_epochs': 44} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\nitzr\AppData\Roaming\Python\Python312\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "c:\Users\nitzr\source\repos\BaseballModels\BaseballModels\Model\Combined\Tuning\../../..\Model\Combined\Tuning\ProWar.py", line 151, in objective
    train_results = TrainAndGraph(
                    ^^^^^^^^^^^^^^
  File "c:\Users\nitzr\source\repos\BaseballModels\BaseballModels\Model\Combined\Tuning\../../..\Model\Combined\Model\Model_Train.py", line 78, in TrainAndGraph
    train_result, test_result = RunEpoch(pro_network, col_network, train_dataset, test_dataset, is_hi

KeyboardInterrupt: 

In [ ]:
print(f"Final best value after refinement: {study.best_value:.4f}")
print(f"Best hyperparameters: {study.best_params}")

In [ ]:
# Code to output a specific trial.  Given run/run randomness, typically a good late trial will be better
# Than one well before it that had better loss when repeated

# trial = study.get_trials()[196]
# print(trial.value)
# print(trial.params)

In [ ]:
import optuna.visualization as vis
vis.plot_param_importances(study).show()

In [ ]:
vis.plot_optimization_history(study).show()

In [ ]:
vis.plot_slice(study).show()